# Verification

See which requirements have calculations, which checks fail, and what remains
missing. This notebook uses the same saved vehicle, flight, and departure inputs
as the first two notebooks. A passing calculation does not close a requirement:
closure needs reviewed evidence for every activity and resolved TBD/TBR values.

In [1]:
import pandas as pd
from IPython.display import display, Markdown

from model.inputs import load_checks, input_tables, choices_table
from model.notebook import show_input_tables, show_table
from model.workspace import current_flight
from model.verification import activities, evaluate, requirement_status
from model.outputs import input_identity, save_verification

## 1. Inputs

[vehicle.toml](inputs/vehicle.toml) and [flight.toml](inputs/flight.toml) define the
shared simulation. [checks.toml](inputs/checks.toml) defines analysis inputs,
limits, and their sources. An activity-specific entry takes precedence over a
shared calculation entry. [requirements](inputs/requirements/) preserve the
imported statements; [evidence](inputs/evidence/) holds supplied review records.

The selection below only changes which rows are displayed; every activity is
still evaluated and exported.

In [2]:
REQUIREMENT = "PERF.2.13"
REQUIREMENT_PREFIX = "AVI.3.COM"

run_inputs = input_identity()
checks = load_checks()
register = activities()
print(f"Imported {register.requirement_id.nunique()} requirements and {len(register)} activities.")

rules = []
for entry, specification in checks.items():
    for rule in specification.get("acceptance", []):
        rules.append({
            "Applies to": entry, "Metric": rule["metric"],
            "Rule": rule["operator"], "Limit": rule["limit"],
            "Units": rule["units"], "Source": rule["source"],
        })
show_table(pd.DataFrame(rules))
show_input_tables(input_tables())

Imported 373 requirements and 677 activities.


Applies to,Metric,Rule,Limit,Units,Source
integrated_tmi,C3_margin_km2_s2,>=,0.00,km^2/s^2,PERF.2.01; target under review
PERF.2.05.A01,liftoff_thrust_to_weight,>=,1.20,1,PERF.2.05
PERF.2.06.A01,max_axial_g,<=,6.00,g,PERF.2.06
PERF.2.13.A02,minimum_control_ratio_margin_Nm,>=,0.00,N m,PERF.2.13 available>=2*aerodynamic moment for all phases
PERF.2.35.A02,fineness_ratio,>=,10.00,1,PERF.2.35
PERF.2.35.A02,fineness_ratio,<=,20.00,1,PERF.2.35
PROP.3.SYS.03.A01,liftoff_thrust_to_weight,>=,1.20,1,PERF.2.05 parent
PERF.2.32.A02,minimum_dry_mass_margin_kg,>=,0.00,kg,PERF.2.32; source top-down allocations
PROP.3.SYS.04.A02,installed_thrust_N,>=,11000000.00,N,PROP.3.SYS.04 [TBR-PROP-02]
PROP.3.SYS.05.A02,installed_thrust_N,>=,890000.00,N,PROP.3.SYS.05 [TBR-PROP-03]


Input,Value,Units,Source
vehicle.diameter_m,5.4,m,inputs/vehicle.toml → vehicle.diameter_m
vehicle.payload_kg,4313,kg,inputs/vehicle.toml → vehicle.payload_kg
vehicle.fairing_kg,2155.27825,kg,inputs/vehicle.toml → vehicle.fairing_kg
vehicle.fairing_length_m,11.5,m,inputs/vehicle.toml → vehicle.fairing_length_m
vehicle.payload_cg_above_fairing_base_m,4.5,m,inputs/vehicle.toml → vehicle.payload_cg_above_fairing_base_m
vehicle.payload_length_m,6.0,m,inputs/vehicle.toml → vehicle.payload_length_m
vehicle.payload_radius_m,1.5,m,inputs/vehicle.toml → vehicle.payload_radius_m
vehicle.fairing_cg_above_base_m,6.5,m,inputs/vehicle.toml → vehicle.fairing_cg_above_base_m
vehicle.payload_station_m,51.763013,m,inputs/vehicle.toml → vehicle.payload_station_m
vehicle.stages.0.name,Stage 1,—,inputs/vehicle.toml → vehicle.stages.0.name


Input,Value,Units,Source
aero.file,inputs/aerodynamics.csv,—,inputs/vehicle.toml → aero.file
aero.reference_area_m2,22.90221,m²,inputs/vehicle.toml → aero.reference_area_m2
aero.cp_station_offset_m,0.0,m,inputs/vehicle.toml → aero.cp_station_offset_m
aero.cp_units,inches converted to m,—,inputs/vehicle.toml → aero.cp_units
aero.cp_datum,nose; RASAero-format inference,—,inputs/vehicle.toml → aero.cp_datum
aero.outside_domain_policy,endpoint hold AND flag every sample,—,inputs/vehicle.toml → aero.outside_domain_policy
aero.post_separation_policy,reuse full vehicle coefficients; unvalidated proxy,—,inputs/vehicle.toml → aero.post_separation_policy


Input,Value,Units,Source
description,Hyperion 28 Oct 2026 synced sizing geometry; selected low-AoA 6DOF RK45 guidance.,—,inputs/flight.toml → description


...

Input,Value,Units,Source
departure.coast_duration_s,600.000000,s,inputs/flight.toml → departure.coast_duration_s
departure.restart_propellant_kg,301.843504,kg,inputs/flight.toml → departure.restart_propellant_kg (uses vehicle.stages.1.startup_propellant_kg)
departure.target_c3_km2_s2,9.800000,km²/s²,inputs/flight.toml → departure.target_c3_km2_s2
departure.max_burn_duration_s,900.000000,s,inputs/flight.toml → departure.max_burn_duration_s
departure.output_step_s,2.000000,s,inputs/flight.toml → departure.output_step_s


Input,Value,Units,Source
shared.minimum_liftoff_thrust_to_weight,1.2,—,inputs/checks.toml → shared.minimum_liftoff_thrust_to_weight


Input,Value,Units,Source
calculations.flight_summary.basis,Current integrated 6DOF ascent; provisional aero/actuator/geometry inputs. No qualification claim.,—,inputs/checks.toml → calculations.flight_summary.basis
calculations.lambert_window.basis,Three source Horizons samples; no daily launch-window optimization or site-access proof.,—,inputs/checks.toml → calculations.lambert_window.basis
calculations.lambert_window.inputs.cases.0.departure_label,2033-04-27 00:00 TDB,—,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)
calculations.lambert_window.inputs.cases.0.arrival_label,2034-01-26 00:00 TDB,—,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)
calculations.lambert_window.inputs.cases.0.time_of_flight_s,23673600.0,s,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)
calculations.lambert_window.inputs.cases.0.departure_position_m,-120881146339.1056; -82360653324.81117; -35699389120.235855,m,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)
calculations.lambert_window.inputs.cases.0.arrival_position_m,151893899759.2209; 144802306802.2028; 62324284364.44935,m,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)
calculations.lambert_window.inputs.cases.0.departure_planet_velocity_m_s,17276.96831291071; -22058.20300105831; -9560.256841566952,m/s,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)
calculations.lambert_window.inputs.cases.0.arrival_planet_velocity_m_s,-16526.05562522464; 17003.911452690558; 8244.829125552214,m/s,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)
calculations.lambert_window.inputs.cases.0.long_way,Yes,—,inputs/checks.toml → calculations.lambert_window.inputs.cases (uses mission_cases.cases)


Input,Value,Units,Source
activities.PERF.2.05.A01.basis,Requirement numeric screen only; model-validation and open source conflicts remain.,—,inputs/checks.toml → activities.PERF.2.05.A01.basis
activities.PERF.2.05.A01.acceptance.0.metric,liftoff_thrust_to_weight,—,inputs/checks.toml → activities.PERF.2.05.A01.acceptance.0.metric
activities.PERF.2.05.A01.acceptance.0.operator,>=,—,inputs/checks.toml → activities.PERF.2.05.A01.acceptance.0.operator
activities.PERF.2.05.A01.acceptance.0.limit,1.2,—,inputs/checks.toml → activities.PERF.2.05.A01.acceptance.0.limit (uses checks.shared.minimum_liftoff_thrust_to_weight)
activities.PERF.2.05.A01.acceptance.0.units,1,—,inputs/checks.toml → activities.PERF.2.05.A01.acceptance.0.units
activities.PERF.2.05.A01.acceptance.0.source,PERF.2.05,—,inputs/checks.toml → activities.PERF.2.05.A01.acceptance.0.source
activities.PERF.2.06.A01.basis,Requirement numeric screen only; model-validation and open source conflicts remain.,—,inputs/checks.toml → activities.PERF.2.06.A01.basis
activities.PERF.2.06.A01.acceptance.0.metric,max_axial_g,—,inputs/checks.toml → activities.PERF.2.06.A01.acceptance.0.metric
activities.PERF.2.06.A01.acceptance.0.operator,<=,—,inputs/checks.toml → activities.PERF.2.06.A01.acceptance.0.operator
activities.PERF.2.06.A01.acceptance.0.limit,6.0,—,inputs/checks.toml → activities.PERF.2.06.A01.acceptance.0.limit (uses limits.axial_g)


Input,Value,Units,Source
studies.tank_thickness.basis,"Pressure-only trade from Structures Sizing.xlsx, Tank Sizing!B3:D8 and Material Properties!B3:H3 (Al2219). The yield factor comes from STR.3.SYS.03; its separate ultimate factor is not evaluated here. Radius follows the selected vehicle. Material applicability, temperature, joints and combined flight loads still need review; this trade alone closes no activity.",—,inputs/checks.toml → studies.tank_thickness.basis
studies.tank_thickness.thicknesses_mm,1; 2; 3; 4; 5; 6,mm,inputs/checks.toml → studies.tank_thickness.thicknesses_mm
studies.tank_thickness.inputs.radius_m,2.7,m,inputs/checks.toml → studies.tank_thickness.inputs.radius_m (uses vehicle.diameter_m × 0.5)
studies.tank_thickness.inputs.pressure_pa,599999.764445,Pa,inputs/checks.toml → studies.tank_thickness.inputs.pressure_pa
studies.tank_thickness.inputs.axial_force_n,0,N,inputs/checks.toml → studies.tank_thickness.inputs.axial_force_n
studies.tank_thickness.inputs.bending_moment_nm,0,N m,inputs/checks.toml → studies.tank_thickness.inputs.bending_moment_nm
studies.tank_thickness.inputs.torque_nm,0,N m,inputs/checks.toml → studies.tank_thickness.inputs.torque_nm
studies.tank_thickness.inputs.yield_allowable_pa,296474563.606224,Pa,inputs/checks.toml → studies.tank_thickness.inputs.yield_allowable_pa
studies.tank_thickness.inputs.joint_efficiency,1,—,inputs/checks.toml → studies.tank_thickness.inputs.joint_efficiency
studies.tank_thickness.inputs.load_factor,1.25,—,inputs/checks.toml → studies.tank_thickness.inputs.load_factor


## 2. Choices and assumptions

| Result | Meaning | Next step |
|---|---|---|
| `SCREEN_PASS` | Calculation meets the supplied limit | Validate assumptions and review evidence |
| `SCREEN_FAIL` | Calculation misses the supplied limit | Review the design, inputs, and limit |
| `CALCULATED` | Result has no acceptance rule | Supply an approved limit and source |
| `MISSING_INPUTS` | Calculation needs more data | Supply the named inputs |
| `EVIDENCE_REQUIRED` | Test, inspection, or analysis record is missing | Add the reviewed evidence |
| `INVALID_INPUTS` / `INVALID_EVIDENCE` | Data or record cannot be used | Fix the stated issue |
| `STALE_EVIDENCE` | Record refers to older inputs or wording | Review it against the current version |

Demonstration data are not vehicle verification evidence. The choices below
remain part of the basis for every simulation-derived result.

In [3]:
show_table(choices_table())

Choice,Selection,Basis
Vehicle geometry,fairing_length_m: 11.5 m; diameter_m: 5.4 m,Selected sizing geometry. The master Payload sheet still gives a 14 m fairing; reconcile with CAD and the aero export.
CP interpretation,cp_units: inches converted to m; cp_datum: nose; RASAero-format inference,Inches are tentative user confirmation; datum is inferred and remains unverified.
Aerodynamic reference,reference_area_m2: 22.902210444669596 m²; outside_domain_policy: endpoint hold AND flag every sample; post_separation_policy: reuse full vehicle coefficients; unvalidated proxy,Source export geometry is unverified. Flag out-of-domain and separated-vehicle proxy samples.
Stage 1 control capability,gimbal_limit_deg: 5.0 deg; gimbal_rate_deg_s: 5.0 deg/s,"Assumed actuator capability, not approved hardware performance."
Stage 2 control capability,gimbal_limit_deg: 5.0 deg; gimbal_rate_deg_s: 5.0 deg/s,"Assumed actuator capability, not approved hardware performance."
Departure baseline,coast_duration_s: 600 s; restart_propellant_kg: 301.843504114127 kg; target_c3_km2_s2: 9.8 km²/s²,The verification target is provisional. The master target is retained as a separate comparison case; neither is a solved encounter.
Integrator,method: RK45; rtol: 2e-07; max_step_s: 0.4 s,Quaternion rigid-body 6DOF with discrete stage and fairing events; no flexible dynamics.
Atmosphere and wind,wind_reference_m_s: 7.62 m/s; wind_toward_deg_east_of_north: 90.0; temperature_offset_K: 0.0 K; density_scale: 1.0,Layered standard atmosphere and a deterministic wind profile; no measured launch sounding.
Coast and roll control,rcs_moment_Nm: 0.0 N m; roll_rcs_moment_Nm: 0.0 N m; control_ratio: 2.0,Evaluate all phases. No central-axis TVC roll authority is available.
Actuator response,actuator_time_constant_s: 0.15 s,Assumed first-order response; requires hardware confirmation.


## 3. Run

This evaluates the full register. It reuses the current ascent and reports
missing inputs or evidence explicitly; it does not fill gaps with example values.

In [4]:
results, details = evaluate(context=current_flight())
requirements = requirement_status(results)

## 4. Outputs

Start with failed checks and unusable records. Full activity and requirement
tables are saved even when only a subset is displayed here.

In [5]:
show_table(results.status.value_counts().rename_axis("Activity result").reset_index(name="Count"))
show_table(requirements.status.value_counts().rename_axis("Requirement result").reset_index(name="Count"))

failure_statuses = ["SCREEN_FAIL", "REVIEWED_FAIL", "INVALID_INPUTS", "INVALID_EVIDENCE"]
failures = results[results.status.isin(failure_statuses)]
columns = {"activity_id": "Activity ID", "activity": "Required work", "status": "Result", "detail": "Basis or issue"}
show_table(failures[list(columns)].rename(columns=columns))

Activity result,Count
EVIDENCE_REQUIRED,431
MISSING_INPUTS,227
CALCULATED,7
SCREEN_FAIL,6
SCREEN_PASS,6


Requirement result,Count
OPEN,367
FAILED_OR_INVALID,6


Activity ID,Required work,Result,Basis or issue
MIS.0.01.A01,Integrated trajectory simulation from launch to TMI.,SCREEN_FAIL,The saved coast and restart debit are screening assumptions. Shared 6DOF prograde TMI; no Mars encounter targeting.
SYS.1.02.A01,"Integrated ascent, coast and finite-burn TMI simulation.",SCREEN_FAIL,The saved coast and restart debit are screening assumptions. Shared 6DOF prograde TMI; no Mars encounter targeting.
SYS.1.09.A02,Integrated finite-burn trajectory simulation.,SCREEN_FAIL,The saved coast and restart debit are screening assumptions. Shared 6DOF prograde TMI; no Mars encounter targeting.
PERF.2.13.A02,Control and aerodynamic moment balance.,SCREEN_FAIL,Requirement numeric screen only; model-validation and open source conflicts remain.
PROP.3.SYS.04.A02,First-stage engine thrust summation.,SCREEN_FAIL,3 source Raptor3 engines at2255kN each; source sea-level basis remains to be substantiated.
PROP.3.TANK.02.A01,Ullage and loaded-liquid volume calculations.,SCREEN_FAIL,"Current Mlox-Hlox Vehicle Sizing G4:H5,F19:G20,B31:C32; Vehicle Mass E29/E44 totals plus fairing2155.27825kg and adapter replacement845−431.976225kg. Bottom-up totals retained to source precision."


### Selected requirement and subsystem

Read the exact wording, source location, and unresolved values before changing
inputs or attaching evidence. Change the view selections in section 1 to inspect
another requirement or subsystem.

In [6]:
selected = results[results.requirement_id.eq(REQUIREMENT)]
if selected.empty:
    print("No matching requirement. See outputs/current/tables/activities.csv for available IDs.")
else:
    requirement = selected.iloc[0]
    display(Markdown(f"**{REQUIREMENT}:** {requirement['statement']}"))
    print("Source:", requirement["source"])
    print("Unresolved values:", requirement["open_parameters"] or "None listed")
    columns = {
        "activity_id": "Activity ID", "activity": "Required work", "kernel": "Calculation",
        "status": "Result", "detail": "Basis or missing inputs",
    }
    show_table(selected[list(columns)].rename(columns=columns).fillna("External evidence"))

subsystem = results[results.requirement_id.str.startswith(REQUIREMENT_PREFIX)]
print("Subsystem selection:", REQUIREMENT_PREFIX)
show_table(subsystem[["activity_id", "activity", "status", "detail"]].rename(columns={
    "activity_id": "Activity ID", "activity": "Required work", "status": "Result", "detail": "Next input or evidence",
}))

**PERF.2.13:** The LV shall maintain a controllability ratio of at least 2.

Source: 2 LV Performance!H14
Unresolved values: None listed


Activity ID,Required work,Calculation,Result,Basis or missing inputs
PERF.2.13.A01,Actuator limit and mass-property inspection.,External evidence,EVIDENCE_REQUIRED,"Add the CAD/FEA, specialist analysis, inspection, demonstration or test record for this activity; see inputs/evidence/README.md."
PERF.2.13.A02,Control and aerodynamic moment balance.,flight_summary,SCREEN_FAIL,Requirement numeric screen only; model-validation and open source conflicts remain.


Subsystem selection: AVI.3.COM


Activity ID,Required work,Result,Next input or evidence
AVI.3.COM.01.A01,RF link budget and antenna-visibility sweep.,MISSING_INPUTS,"tx_positions_m, rx_position_m, frequency_hz, tx_power_w, tx_gain_dbi, rx_gain_dbi, loss_db, noise_temperature_k, bit_rate_bps, required_ebn0_db, minimum_elevation_deg"
AVI.3.COM.01.A02,Radio-frequency and link-performance tests.,EVIDENCE_REQUIRED,"Add the CAD/FEA, specialist analysis, inspection, demonstration or test record for this activity; see inputs/evidence/README.md."
AVI.3.COM.02.A01,Telemetry packet-rate and bandwidth budget.,MISSING_INPUTS,"streams, link_rate_bps, coding_rate, protocol_efficiency, propagation_delay_s, processing_delay_s"
AVI.3.COM.02.A02,Telemetry continuity functional tests.,EVIDENCE_REQUIRED,"Add the CAD/FEA, specialist analysis, inspection, demonstration or test record for this activity; see inputs/evidence/README.md."
AVI.3.COM.03.A01,Engine-command telemetry packet-rate budget.,MISSING_INPUTS,"streams, link_rate_bps, coding_rate, protocol_efficiency, propagation_delay_s, processing_delay_s"
AVI.3.COM.03.A02,Command-to-telemetry end-to-end tests.,EVIDENCE_REQUIRED,"Add the CAD/FEA, specialist analysis, inspection, demonstration or test record for this activity; see inputs/evidence/README.md."
AVI.3.COM.04.A01,Link-recovery state-transition and timing model.,MISSING_INPUTS,"state_inputs, timing_inputs, fault_inputs"
AVI.3.COM.04.A02,Communication-outage fault-injection tests.,EVIDENCE_REQUIRED,"Add the CAD/FEA, specialist analysis, inspection, demonstration or test record for this activity; see inputs/evidence/README.md."
AVI.3.COM.05.A01,GNSS satellite-visibility geometry sweep.,MISSING_INPUTS,"receiver_positions_m, satellite_positions_m, minimum_elevation_deg, earth_radius_m"
AVI.3.COM.05.A02,GNSS receiver update-rate performance tests.,EVIDENCE_REQUIRED,"Add the CAD/FEA, specialist analysis, inspection, demonstration or test record for this activity; see inputs/evidence/README.md."


## 5. What to do next

1. Find the activity in the saved [activity table](outputs/current/tables/activity_results.csv).
2. Find its arguments in [calculation_inputs.csv](outputs/current/tables/calculation_inputs.csv).
3. Add sourced inputs and acceptance rules to [checks.toml](inputs/checks.toml).
4. Add reviewed tests, inspections, or specialist analyses using the [evidence guide](inputs/evidence/README.md).
5. Run All again. Read the [summary](outputs/current/summary.md) and [requirement status](outputs/current/tables/requirement_status.csv).

For a requirements revision, follow the [README import steps](README.md).
The importer reads a saved workbook; it does not synchronize or edit OneDrive.
Each export includes the [exact inputs used](outputs/current/inputs_used/).

In [7]:
output_folder = save_verification(results, details, expected_inputs=run_inputs)
print(f"Saved {len(results)} activities and {len(requirements)} requirements to:", output_folder)

Saved 677 activities and 373 requirements to: C:\Users\maxwe\.codex\.chatgpt-projects\g-p-6a9464860c90819188ae3dd3be539576\Hyperion\outputs\current
